In [9]:
import os
import re
import urllib.request
from pathlib import Path
from typing import List, Optional, Tuple

import cv2
import joblib
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

DATA_DIR = Path("data")
MODEL_PATH = Path("model.pkl")

POSE_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "pose_landmarker/pose_landmarker_lite/float16/latest/"
    "pose_landmarker_lite.task"
)
POSE_MODEL_PATH = Path("pose_landmarker_lite.task")

if not POSE_MODEL_PATH.exists():
    print("[INFO] Downloading pose_landmarker_lite.task ...")
    urllib.request.urlretrieve(POSE_MODEL_URL, str(POSE_MODEL_PATH))
    print("[OK] Model downloaded.")

JUMP_LABEL = 1
BEND_LABEL = 0

print("Data dir:", DATA_DIR.resolve())
print("Model output:", MODEL_PATH.resolve())
print("Pose model:", POSE_MODEL_PATH.resolve())

Data dir: C:\workspace\CNDPT3_N8\data
Model output: C:\workspace\CNDPT3_N8\model.pkl
Pose model: C:\workspace\CNDPT3_N8\pose_landmarker_lite.task


In [10]:
POSE_LANDMARK_COUNT = 33
KEYPOINT_DIM = 3
EXPECTED_FEATURE_SIZE = POSE_LANDMARK_COUNT * KEYPOINT_DIM

video_pattern = re.compile(r"^(jump|bend|jumb)_(\d+)\.mp4$", re.IGNORECASE)


def parse_video_info(path: Path) -> Optional[Tuple[str, int]]:
    m = video_pattern.match(path.name)
    if not m:
        return None
    cls_raw = m.group(1).lower()
    idx = int(m.group(2))
    cls = "jump" if cls_raw in {"jump", "jumb"} else "bend"
    return cls, idx


def list_labeled_videos(data_dir: Path):
    items = []
    for p in sorted(data_dir.glob("*.mp4")):
        info = parse_video_info(p)
        if info is None:
            continue
        cls, idx = info
        label = JUMP_LABEL if cls == "jump" else BEND_LABEL
        items.append({"path": p, "cls": cls, "idx": idx, "label": label})
    return items


videos = list_labeled_videos(DATA_DIR)
print(f"Found {len(videos)} usable videos")
print("Sample:", [v["path"].name for v in videos[:8]])

Found 34 usable videos
Sample: ['bend_1.mp4', 'bend_10.mp4', 'bend_2.mp4', 'bend_3.mp4', 'bend_4.mp4', 'bend_5.mp4', 'bend_6.mp4', 'bend_7.mp4']


In [11]:
def create_pose_landmarker():
    """Create a PoseLandmarker using Tasks API (VIDEO mode)."""
    with open(str(POSE_MODEL_PATH), "rb") as f:
        model_data = f.read()
    base_options = mp_python.BaseOptions(model_asset_buffer=model_data)
    options = mp_vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=mp_vision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return mp_vision.PoseLandmarker.create_from_options(options)


def extract_video_feature(video_path: Path) -> Optional[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"[WARN] Cannot open: {video_path.name}")
        return None

    landmarker = create_pose_landmarker()
    frame_features: List[np.ndarray] = []
    frame_idx = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        frame_idx += 1
        timestamp_ms = frame_idx * 33

        result = landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.pose_landmarks and len(result.pose_landmarks) > 0:
            landmarks = result.pose_landmarks[0]
            coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks], dtype=np.float32).reshape(-1)
            if coords.size == EXPECTED_FEATURE_SIZE:
                frame_features.append(coords)

    cap.release()
    landmarker.close()

    if not frame_features:
        print(f"[WARN] No pose detected: {video_path.name}")
        return None

    arr = np.stack(frame_features, axis=0)
    mean_feat = arr.mean(axis=0)
    std_feat = arr.std(axis=0)
    min_feat = arr.min(axis=0)
    max_feat = arr.max(axis=0)

    feat = np.concatenate([mean_feat, std_feat, min_feat, max_feat], axis=0)
    mu = feat.mean()
    sigma = feat.std() + 1e-8
    feat = (feat - mu) / sigma
    return feat.astype(np.float32)

In [12]:
def split_samples_80_20(samples):
    train, test = [], []
    for cls_name in ["bend", "jump"]:
        cls_samples = sorted([s for s in samples if s["cls"] == cls_name], key=lambda x: x["idx"])
        n = len(cls_samples)
        if n == 0:
            continue

        if n == 1:
            n_train = 1
        else:
            n_train = int(round(n * 0.8))
            n_train = min(max(1, n_train), n - 1)

        train.extend(cls_samples[:n_train])
        test.extend(cls_samples[n_train:])

    return train, test


all_samples = []

for i, v in enumerate(videos):
    print(f"[{i+1}/{len(videos)}] Processing {v['path'].name} ...", end=" ")
    feat = extract_video_feature(v["path"])
    if feat is None:
        continue
    all_samples.append({**v, "feature": feat})
    print("OK")

print(f"\nExtracted features: {len(all_samples)} videos")

train_samples, test_samples = split_samples_80_20(all_samples)

print("Train videos:", len(train_samples))
print("Test videos:", len(test_samples))
print("Train class count:", {"bend": sum(s["cls"] == "bend" for s in train_samples), "jump": sum(s["cls"] == "jump" for s in train_samples)})
print("Test class count:", {"bend": sum(s["cls"] == "bend" for s in test_samples), "jump": sum(s["cls"] == "jump" for s in test_samples)})

if len(train_samples) == 0 or len(test_samples) == 0:
    raise RuntimeError("Not enough data after feature extraction. Add videos and rerun.")

[1/34] Processing bend_1.mp4 ... OK
[2/34] Processing bend_10.mp4 ... OK
[3/34] Processing bend_2.mp4 ... OK
[4/34] Processing bend_3.mp4 ... OK
[5/34] Processing bend_4.mp4 ... OK
[6/34] Processing bend_5.mp4 ... OK
[7/34] Processing bend_6.mp4 ... OK
[8/34] Processing bend_7.mp4 ... OK
[9/34] Processing bend_8.mp4 ... OK
[10/34] Processing bend_9.mp4 ... OK
[11/34] Processing jump_1.mp4 ... OK
[12/34] Processing jump_10.mp4 ... OK
[13/34] Processing jump_11.mp4 ... OK
[14/34] Processing jump_12.mp4 ... OK
[15/34] Processing jump_13.mp4 ... OK
[16/34] Processing jump_14.mp4 ... OK
[17/34] Processing jump_15.mp4 ... OK
[18/34] Processing jump_16.mp4 ... OK
[19/34] Processing jump_17.mp4 ... OK
[20/34] Processing jump_18.mp4 ... OK
[21/34] Processing jump_19.mp4 ... OK
[22/34] Processing jump_2.mp4 ... OK
[23/34] Processing jump_20.mp4 ... OK
[24/34] Processing jump_21.mp4 ... OK
[25/34] Processing jump_22.mp4 ... OK
[26/34] Processing jump_23.mp4 ... OK
[27/34] Processing jump_24.mp4 .

In [13]:
X_train = np.stack([s["feature"] for s in train_samples], axis=0)
y_train = np.array([s["label"] for s in train_samples], dtype=np.int32)

X_test = np.stack([s["feature"] for s in test_samples], axis=0)
y_test = np.array([s["label"] for s in test_samples], dtype=np.int32)
test_names = [s["path"].name for s in test_samples]

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

X_train: (27, 396) y_train: (27,)
X_test: (7, 396) y_test: (7,)


In [14]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    class_weight="balanced",
)

model.fit(X_train, y_train)
print("Model trained.")

Model trained.


In [15]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print("Confusion Matrix [rows=true, cols=pred]:")
print(cm)
print("\\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Bend", "Jump"]))

print("\\nPer-video predictions:")
for name, pred, prob in zip(test_names, y_pred, y_proba):
    confidence = float(prob[pred])
    pred_name = "Jump" if pred == JUMP_LABEL else "Bend"
    print(f"Video: {name} -> Predict: {pred_name} (Confidence: {confidence:.2f})")

Accuracy: 1.0000
Confusion Matrix [rows=true, cols=pred]:
[[2 0]
 [0 5]]
\nClassification report:
              precision    recall  f1-score   support

        Bend       1.00      1.00      1.00         2
        Jump       1.00      1.00      1.00         5

    accuracy                           1.00         7
   macro avg       1.00      1.00      1.00         7
weighted avg       1.00      1.00      1.00         7

\nPer-video predictions:
Video: bend_9.mp4 -> Predict: Bend (Confidence: 0.95)
Video: bend_10.mp4 -> Predict: Bend (Confidence: 0.63)
Video: jump_20.mp4 -> Predict: Jump (Confidence: 1.00)
Video: jump_21.mp4 -> Predict: Jump (Confidence: 0.99)
Video: jump_22.mp4 -> Predict: Jump (Confidence: 1.00)
Video: jump_23.mp4 -> Predict: Jump (Confidence: 0.99)
Video: jump_24.mp4 -> Predict: Jump (Confidence: 0.94)


In [16]:
metadata = {
    "model": model,
    "feature_type": "mediapipe33_xyz_mean_std_min_max_per_video",
    "feature_size": int(X_train.shape[1]),
    "labels": {"bend": BEND_LABEL, "jump": JUMP_LABEL},
}
joblib.dump(metadata, MODEL_PATH)
print(f"Saved: {MODEL_PATH.resolve()}")

Saved: C:\workspace\CNDPT3_N8\model.pkl
